In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")

print("Dataset loaded successfully")
print("Rows and Columns:", df.shape)
display(df.head())

Dataset loaded successfully
Rows and Columns: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Task 3: Data Cleaning

## 1. Data Quality Report

This section examines the original dataset for missing values, duplicate rows, incorrect data types, and unusual numeric values before cleaning.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nNull Count Per Column:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nNumeric Summary:")
display(df.describe().T)

Dataset Shape: (891, 12)

First 5 Rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Column Names:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Null Count Per Column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicate Rows: 0

Numeric Summary:


,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292


In [3]:
original_df = df.copy()

before_rows = len(df)
before_nulls = int(df.isnull().sum().sum())
before_duplicates = int(df.duplicated().sum())
before_dtypes = df.dtypes.astype(str).to_dict()

print("Original rows:", before_rows)
print("Total missing values:", before_nulls)
print("Duplicate rows:", before_duplicates)

Original rows: 891
Total missing values: 866
Duplicate rows: 0


## 2. Missing Data Handling

- Age: Missing values are replaced with the median age because the median is less sensitive to extreme values than the mean.
- Cabin: Missing values are replaced with "Unknown" because most cabin entries are missing, and deleting these rows would discard useful passenger information.
- Embarked: Missing values are replaced with the mode because this is a categorical column and the most frequent port provides a reasonable replacement.
- Other columns: No missing-value imputation is required because they contain no missing values in the initial report.

These choices preserve the dataset rows while making the data suitable for further analysis.

In [4]:
age_median = df["Age"].median()
embarked_mode = df["Embarked"].mode()[0]

df["Age"] = df["Age"].fillna(age_median)
df["Cabin"] = df["Cabin"].fillna("Unknown")
df["Embarked"] = df["Embarked"].fillna(embarked_mode)

print("Missing values after imputation:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Missing values after imputation:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

Total missing values: 0


In [5]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates().copy()

duplicates_after = df.duplicated().sum()
rows_removed = before_rows - len(df)

print("Duplicate rows before cleaning:", duplicates_before)
print("Duplicate rows removed:", rows_removed)
print("Duplicate rows after cleaning:", duplicates_after)
print("Rows remaining:", len(df))

Duplicate rows before cleaning: 0
Duplicate rows removed: 0
Duplicate rows after cleaning: 0
Rows remaining: 891


## 3. Duplicate Removal and Standardisation

Exact duplicate rows were identified and removed using pandas. No duplicate rows were present, so all 891 rows were retained.

Text columns were standardised by trimming unnecessary whitespace. Gender values were converted to consistent title-case formatting: "male" to "Male" and "female" to "Female". The original meaning of the data was preserved.

In [6]:
text_columns = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

df["Sex"] = df["Sex"].str.lower().map({
    "male": "Male",
    "female": "Female"
})

df["Embarked"] = df["Embarked"].str.upper()

print("Sex values:", df["Sex"].unique())
print("Embarked values:", df["Embarked"].unique())
print("Extra spaces checked and removed from text columns.")

Sex values: <StringArray>
['Male', 'Female']
Length: 2, dtype: str
Embarked values: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
Extra spaces checked and removed from text columns.


## 4. Outlier Detection Using the IQR Method

The Interquartile Range (IQR) method is used to identify potential outliers in numeric columns.

IQR = Q3 - Q1

Lower Bound = Q1 - 1.5 × IQR

Upper Bound = Q3 + 1.5 × IQR

Values outside these bounds are flagged as potential outliers. PassengerId is excluded because it is an identifier, and Survived is excluded because it is a binary target. Pclass is categorical, so it is also excluded from outlier analysis.

Outliers in Age, SibSp, Parch, and Fare will be retained unless they are confirmed to be data-entry errors. Extreme values may represent genuine passenger characteristics, family sizes, or ticket prices. Automatically deleting or capping them could remove useful information.

In [7]:
numeric_cols = ["Age", "SibSp", "Parch", "Fare"]

outlier_results = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    outlier_results.append({
        "Column": col,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Outlier Count": len(outliers),
        "Decision": "Retain and document"
    })

outlier_report = pd.DataFrame(outlier_results)
display(outlier_report)

,Column,Lower Bound,Upper Bound,Outlier Count,Decision
0,Age,2.50,54.50,66,Retain and document
1,SibSp,-1.50,2.50,46,Retain and document
2,Parch,0.00,0.00,213,Retain and document
3,Fare,-26.72,65.63,116,Retain and document


In [8]:
df["PassengerId"] = df["PassengerId"].astype("string")
df["Survived"] = df["Survived"].astype("int64")
df["Pclass"] = df["Pclass"].astype("int64")
df["Name"] = df["Name"].astype("string")
df["Sex"] = df["Sex"].astype("string")
df["Age"] = pd.to_numeric(df["Age"], errors="raise").astype("float64")
df["SibSp"] = df["SibSp"].astype("int64")
df["Parch"] = df["Parch"].astype("int64")
df["Ticket"] = df["Ticket"].astype("string")
df["Fare"] = pd.to_numeric(df["Fare"], errors="raise").astype("float64")
df["Cabin"] = df["Cabin"].astype("string")
df["Embarked"] = df["Embarked"].astype("string")

print("Corrected Data Types:")
print(df.dtypes)

Corrected Data Types:
PassengerId     string
Survived         int64
Pclass           int64
Name            string
Sex             string
Age            float64
SibSp            int64
Parch            int64
Ticket          string
Fare           float64
Cabin           string
Embarked        string
dtype: object


## 5. Before vs. After Cleaning Summary

This section compares the original dataset with the cleaned dataset using row count, total missing values, duplicate count, and data type validation.

All missing values were handled using the documented strategies. Duplicate rows were checked and removed if present. Text formatting was standardised, data types were corrected, and potential outliers were retained because they may represent genuine passenger values.

In [9]:
before_summary = {
    "Stage": "Before Cleaning",
    "Rows": len(original_df),
    "Total Null Values": int(original_df.isnull().sum().sum()),
    "Duplicate Rows": int(original_df.duplicated().sum()),
    "Dtype Validation": "Initial types inspected"
}

after_summary = {
    "Stage": "After Cleaning",
    "Rows": len(df),
    "Total Null Values": int(df.isnull().sum().sum()),
    "Duplicate Rows": int(df.duplicated().sum()),
    "Dtype Validation": "Corrected and verified"
}

summary_df = pd.DataFrame([before_summary, after_summary])
display(summary_df)

,Stage,Rows,Total Null Values,Duplicate Rows,Dtype Validation
0,Before Cleaning,891,866,0,Initial types inspected
1,After Cleaning,891,0,0,Corrected and verified


In [10]:
output_file = "Titanic_Cleaned.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully:", output_file)
print("Final dataset shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())
print("Remaining duplicate rows:", df.duplicated().sum())

display(df.head())

Cleaned dataset saved successfully: Titanic_Cleaned.csv
Final dataset shape: (891, 12)
Remaining missing values: 0
Remaining duplicate rows: 0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.2500,Unknown,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.9250,Unknown,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.0500,Unknown,S
